# Landing — trust price history

Source: `data/uk_investment_trusts_price_history_monthly.csv` — monthly, price only,
no dividends. Lands **exactly as it arrived**, known problems included:

- date labels mix month-end and next-month-first (Dec 31 shows as Jan 1)
- ~25 tickers carry interleaved rows on a wrong scale
- one zero price (`PCFT`, 2019-11-01)

All of that is Silver's job. Landing must leave it alone.

Expected: **16,357 rows**.

In [0]:
import os

import pandas as pd

CATALOG = "`index-vs-trust-pipeline`"
TABLE = f"{CATALOG}.landing.trust_prices_raw"

REPO_ROOT = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
CSV_PATH = os.path.join(
    REPO_ROOT, "data", "uk_investment_trusts_price_history_monthly.csv"
)

print(f"reading {CSV_PATH}")

In [0]:
# Text in, text out -- the price column stays a string so a malformed number survives
# to Silver instead of becoming a null at the front door.
prices = pd.read_csv(CSV_PATH, dtype=str, keep_default_na=False)

print(f"{len(prices)} rows, {len(prices.columns)} columns")
print(list(prices.columns))
prices.head(3)

In [0]:
sdf = spark.createDataFrame(prices)
sdf.write.format("delta").mode("overwrite").saveAsTable(TABLE)

print(f"wrote {TABLE}")

## Verification

In [0]:
%sql
SELECT COUNT(*) AS row_count,
       COUNT(DISTINCT ticker) AS distinct_tickers,
       MIN(`date`) AS first_date,
       MAX(`date`) AS last_date
FROM `index-vs-trust-pipeline`.landing.trust_prices_raw;

Expect **16,357 rows, 102 tickers, 2011-09-30 to 2026-09-17**.

In [0]:
%sql
-- The known-bad row must still be bad. This proves Landing cleaned nothing.
SELECT ticker, `date`, price_gbx_or_gbp
FROM `index-vs-trust-pipeline`.landing.trust_prices_raw
WHERE ticker = 'PCFT' AND `date` = '2019-11-01';